In [1]:
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec

# -----------------------------
# CONFIG
# -----------------------------
BASE = Path("../benchmarks")  # change if needed

D_FOLDERS = {
    0: BASE / "cifar100_all" / "debug_vis",
    1: BASE / "cifar100_all_d_1" / "debug_vis",
    2: BASE / "cifar100_all_d_2" / "debug_vis",
    3: BASE / "cifar100_all_d_3" / "debug_vis",
    4: BASE / "cifar100_all_d_4" / "debug_vis",
}

OUT_DIR = BASE / "cifar100_masks_one_score_strips"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_FROM_D = 1     # <-- score shown next to original (set 4 if you want d4)
N_PER_FIG = 20
MAX_ITEMS = 200
DPI = 450

# Tight layout
WSPACE = 0.01
HSPACE = 0.02
MARGINS = dict(left=0.03, right=0.995, top=0.98, bottom=0.02)

# Column widths: Original a bit wider than masks
W_ORIG = 1.70
W_MASK = 0.55

# -----------------------------
# Parsing
# -----------------------------
PAT = re.compile(r"__label_(\d+)__idx_(\d+)_(.+)\.png")
SCORE_PAT = re.compile(r"score_(\d*\.?\d+)")

def parse_file(p: Path):
    m = PAT.search(p.name)
    if not m:
        return None
    label = int(m.group(1))
    idx = int(m.group(2))
    rest = m.group(3)

    if rest == "original":
        return (label, idx, "original", None)
    if "mask_bw" in rest:
        return (label, idx, "mask", None)
    if "score_" in rest:
        sm = SCORE_PAT.search(rest)
        score = float(sm.group(1)) if sm else None
        return (label, idx, "score", score)
    return None

def index_folder(folder: Path):
    data = {}
    for p in folder.glob("*.png"):
        parsed = parse_file(p)
        if parsed is None:
            continue
        label, idx, kind, score = parsed
        key = (label, idx)
        if key not in data:
            data[key] = {"original": None, "mask": None, "score": None, "score_val": None}
        data[key][kind] = p
        if kind == "score":
            data[key]["score_val"] = score
    return data

def safe_read(p: Path):
    try:
        return mpimg.imread(p)
    except Exception:
        return None

# -----------------------------
# Build indices and common keys
# -----------------------------
idx_by_d = {d: index_folder(folder) for d, folder in D_FOLDERS.items()}

common = None
for d in sorted(D_FOLDERS):
    keys = {k for k, v in idx_by_d[d].items() if v.get("original") and v.get("mask") and v.get("score")}
    common = keys if common is None else (common & keys)

items = sorted(list(common), key=lambda k: (k[0], k[1]))
if MAX_ITEMS is not None:
    items = items[:MAX_ITEMS]

print("Instances found:", len(items))

# -----------------------------
# Render: Orig(+one score) | d1 mask | d2 mask | d3 mask | d4 mask
# -----------------------------
def render_batch(batch_keys, out_path: Path):
    n = len(batch_keys)
    ncols = 1 + len(D_FOLDERS)

    width_ratios = [W_ORIG] + [W_MASK] * len(D_FOLDERS)
    fig_w = sum(width_ratios) * 1.0
    fig_h = max(1.0, n * 0.52)

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = GridSpec(n, ncols, figure=fig, width_ratios=width_ratios)
    fig.subplots_adjust(wspace=WSPACE, hspace=HSPACE, **MARGINS)

    headers = ["Orig (score)"] + [f"d{d} mask" for d in sorted(D_FOLDERS)]

    for r, key in enumerate(batch_keys):
        label, idx = key

        # --- Original cell ---
        ax0 = fig.add_subplot(gs[r, 0])
        orig_img = safe_read(idx_by_d[1][key]["original"])
        if orig_img is not None:
            ax0.imshow(orig_img)
        ax0.axis("off")

        # Show ONE score (from SCORE_FROM_D) next to original
        sc = idx_by_d[SCORE_FROM_D][key]["score_val"]
        ax0.text(
            1.02, 0.50, f"d{SCORE_FROM_D}: {sc:.3f}",
            transform=ax0.transAxes,
            ha="left", va="center",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.80, pad=1, edgecolor="none"),
            clip_on=False
        )

        # optional row id (comment out if you don't want)
        ax0.text(
            -0.03, 0.5, f"L{label}\n#{idx}",
            transform=ax0.transAxes,
            ha="right", va="center",
            fontsize=6,
            clip_on=False
        )

        # --- Masks for d=1..4 ---
        for j, d in enumerate(sorted(D_FOLDERS), start=1):
            axm = fig.add_subplot(gs[r, j])
            mimg = safe_read(idx_by_d[d][key]["mask"])
            if mimg is not None:
                axm.imshow(mimg)
            axm.axis("off")

        # headers only first row
        if r == 0:
            # first-row axes are created first; set titles via direct access
            for col in range(ncols):
                fig.axes[col].set_title(headers[col], fontsize=7, pad=1)

    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)

for start in range(0, len(items), N_PER_FIG):
    batch = items[start:start + N_PER_FIG]
    out_path = OUT_DIR / f"mask_strip_one_score_{start:04d}.png"
    render_batch(batch, out_path)

print("Saved to:", OUT_DIR)

Instances found: 0
Saved to: ../benchmarks/cifar100_masks_one_score_strips
